In [ ]:
import numpy
import random
import uuid
from algorithms.models import RPDHGParams, SolverConfig, GurobiParams
from algorithms.pdhg_restarted_cu import pdhg_restarted_cu
from algorithms.gurobi_lp import solve_lp_gurobi
from experiments.evaluation import evaluate_solver
from pathlib import Path
from dataclasses import replace
import transportation_problems
from transportation_problems.dotmark.loader_csv import load_dotmark_instance_csv
from transportation_problems.dotmark.build_ot_problem import build_ot_lp

def get_string_tuple_list(folder:str):
    string_Y = "transportation_problems/dotmark/csv_data/FOLDER/data32_XXXX.csv"
    string_X = string_Y.replace("FOLDER", folder)
    
    number_tuple_list = []
    for i in range(1,11):
        for j in range(i +1,11):
            number_tuple_list.append((i +1000, j+1000))
    
    string_tuple_list = []
    for tpl in number_tuple_list:
        string_A = string_X.replace("XXXX", str(tpl[0]))
        string_D = string_X.replace("XXXX", str(tpl[1]))
        string_tuple_list.append((string_A, string_D))
        
    return string_tuple_list

folder_names = ["LogGRF","MicroscopyImages","WhiteNoise","CauchyDensity","ClassicImages","GRFmoderate","GRFrough","GRFsmooth","LogitGRF","Shapes"]

for folder_name in folder_names:

    for x in get_string_tuple_list(folder_name):
        path_A = Path(x[0])
        path_B = Path(x[1])

        print("Loading images...")
        Aimg, Bimg, a, b = load_dotmark_instance_csv(path_A, path_B)
        H, W = Aimg.shape

        print("Building OT LP...")

        from_number = x[0].split("/").pop()
        to_number = x[1].split("/").pop()

        good_name = folder_name + str(from_number) + "_to_" + str(to_number)

        dotmark_problem = build_ot_lp(a, b, H, W, name = good_name)

        print("LP size:", dotmark_problem.A.shape)


        tolerance = 1e-8
        my_params1 = RPDHGParams(
                        tau = None,
                        sigma = None,
                        theta = 1.01,
                        alpha= 1.0,
                        rebalancing_threshhold = 3.0,
                        step_shrinkage = 0.75,
                        restart_check = "adaptive" ,   #"adaptive" | "fixed" | "none"
                        min_epoch_length = 250,
                        max_iter = 100_000,
                        fixed_iter_restart= 3000,
                        tol_primal = tolerance,
                        tol_dual = tolerance,
                        tol_gap = tolerance,
                        tau_sigma_preconditioned= True,
                        rebalance_tau_sigma= True,
                        diagnostik_i = 25
        )


        directory_name = r"experiments\32 theta"


        gurobi_params = GurobiParams()
        evaluate_solver(
                problem=dotmark_problem,
                solver_fn= lambda dotmark_problem: solve_lp_gurobi(dotmark_problem,gurobi_params),
                write_run=True,
                csv_export= False,
                exp_dir=directory_name,
                experiment_name=str(uuid.uuid4()),
                solver_config= SolverConfig(solver_name= "gurobi", params = gurobi_params)
            )

        my_params2 = replace(my_params1,theta=1.03)
        my_params3 = replace(my_params1,theta=1.05)
        params = [my_params1,my_params2,my_params3]
        for p in params:

            evaluate_solver(
                problem=dotmark_problem,
                solver_fn= lambda dotmark_problem: pdhg_restarted_cu(dotmark_problem,p),
                write_run=True,
                csv_export= False,
                exp_dir=directory_name,
                experiment_name=str(uuid.uuid4()),
                solver_config= SolverConfig(solver_name= "theta" + str(p.theta), params = p)
            )

